# 人工智能图像数据预处理与增强实操 · 教学参考

这是帮助理解写法的参考实现，不是官方标准答案。请在 `answer.ipynb` 独立作答；不要在本文件中覆盖自己的草稿。

In [ ]:
from pathlib import Path
import random
from PIL import Image, ImageEnhance

INPUT_DIR = Path("raw_images")
OUTPUT_DIR = Path("augmented_images")
TARGET_SIZE = 512

def rename_images(input_dir):
    files = sorted(path for path in input_dir.iterdir() if path.suffix.lower() == ".jpg")
    renamed = []
    for index, path in enumerate(files, start=1):
        new_path = input_dir / f"insulator_defect_{index:04d}.jpg"
        if path != new_path:
            path.rename(new_path)
        renamed.append(new_path)
    return renamed

def pad_to_minimum(image):
    width, height = image.size
    if width < TARGET_SIZE or height < TARGET_SIZE:
        canvas = Image.new("RGB", (max(width, TARGET_SIZE), max(height, TARGET_SIZE)), (0, 0, 0))
        canvas.paste(image, ((canvas.width - width) // 2, (canvas.height - height) // 2))
        return canvas
    return image

def augment(image):
    image = pad_to_minimum(image)
    left = random.randint(0, image.width - TARGET_SIZE)
    top = random.randint(0, image.height - TARGET_SIZE)
    image = image.crop((left, top, left + TARGET_SIZE, top + TARGET_SIZE))
    angle = random.uniform(-15.0, 15.0)
    image = image.rotate(angle, resample=Image.Resampling.BICUBIC, fillcolor=(0, 0, 0))
    image = ImageEnhance.Brightness(image).enhance(random.uniform(0.8, 1.2))
    image = ImageEnhance.Contrast(image).enhance(1.5)
    return image

def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for path in rename_images(INPUT_DIR):
        try:
            with Image.open(path) as image:
                result = augment(image.convert("RGB"))
                output_path = OUTPUT_DIR / f"{path.stem}_aug.jpg"
                result.save(output_path, quality=95)
                print(f"已保存：{output_path}")
        except OSError as error:
            print(f"跳过 {path.name}：{error}")

if __name__ == "__main__":
    main()
